# Fase 1: probar prompts con distintos tonos

Este notebook recorre el flujo de principio a fin sin entrenar el modelo. Usaremos reseñas de Hugging Face, construiremos instrucciones de resumen y compararemos las respuestas de FLAN-T5 en tono neutral, friendly y formal.

**Antes de empezar:** activa el entorno virtual del proyecto e instala PyTorch con la variante CUDA recomendada para tu sistema, además de `requirements.txt`. La primera carga puede descargar el dataset y el modelo desde Hugging Face; necesitas conexión a internet. La prueba está limitada a dos reseñas para mantenerla corta.

## 1. Encontrar el proyecto y leer su configuración

El notebook puede abrirse desde la carpeta del repositorio o desde `notebooks/`. Esta celda busca hacia arriba hasta encontrar `src/phase1_prompting.py`. Después lee `configs/default.yaml`, donde se definen el nombre del dataset, sus columnas, el modelo y los límites de generación.

In [ ]:
import sys
from pathlib import Path

possible_roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    (path for path in possible_roots if (path / 'src' / 'phase1_prompting.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('No encuentro la raíz del repositorio. Abre el notebook desde la carpeta del proyecto.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.phase1_prompting import build_prompt, generate_summary, load_config
from src.data import load_reviews

CONFIG = load_config(PROJECT_ROOT / 'configs' / 'default.yaml')
DATA_CONFIG = CONFIG['data']
PHASE_CONFIG = CONFIG['phase1']
MODEL_NAME = CONFIG['project']['model_name']

print('Raíz del proyecto:', PROJECT_ROOT)
print('Modelo:', MODEL_NAME)
print('Dataset y variante:', DATA_CONFIG['dataset_name'], DATA_CONFIG.get('dataset_config'))
print('Columnas:', DATA_CONFIG['text_column'], DATA_CONFIG.get('reference_column'), DATA_CONFIG.get('rating_column'))
print('Tonos:', PHASE_CONFIG['tone_labels'])

## 2. Cargar unas pocas reseñas

`load_reviews` usa `datasets.load_dataset` para localizar el dataset por su identificador en Hugging Face Hub y descargarlo si aún no está en caché. `test` es la partición de prueba. Aquí tomamos como máximo dos filas para explorar; las reseñas sin texto se descartan. El título se guarda como referencia informativa, no como resumen correcto.

In [ ]:
N_SAMPLES = 2  # Increase this number to explore more examples.
reviews = list(
    load_reviews(
        dataset_name=DATA_CONFIG['dataset_name'],
        dataset_config=DATA_CONFIG.get('dataset_config'),
        split='test',
        text_column=DATA_CONFIG['text_column'],
        reference_column=DATA_CONFIG.get('reference_column'),
        rating_column=DATA_CONFIG.get('rating_column'),
        max_samples=N_SAMPLES,
    )
)

if not reviews:
    raise ValueError('No se encontraron reseñas con texto en las filas solicitadas.')

print(f'Reseñas cargadas: {len(reviews)}')
print('Valoración:', reviews[0].get('rating', 'no disponible'))
print('Título informativo:', reviews[0].get('reference_title', 'no disponible'))
print('Texto de la reseña:', reviews[0]['review_text'][:700])

## 3. Construir los prompts y entender los tokens

Un prompt es la instrucción completa que recibe el modelo: tarea, tono y reseña. Solo cambiaremos la frase del tono para que la comparación sea razonable.

`max_input_tokens` limita esa entrada completa después de que el tokenizer la convierte en piezas de texto llamadas tokens. No son exactamente palabras. Si la instrucción más la reseña excede el límite, el tokenizer recorta la entrada. `max_new_tokens` limita por separado la longitud máxima de la respuesta generada.

In [ ]:
prompts = {
    tone: build_prompt(reviews[0]['review_text'], tone)
    for tone in PHASE_CONFIG['tone_labels']
}

for tone, prompt in prompts.items():
    print(f'--- Tono: {tone} ---')
    print(prompt)
    print()

## 4. Cargar FLAN-T5 una vez

El tokenizer transforma texto en tokens y el modelo produce tokens de respuesta. Ambos se descargan de Hugging Face si no están en la caché. Si PyTorch detecta CUDA, usaremos la GPU y precisión de 16 bits para reducir memoria; de lo contrario, usaremos CPU, que puede tardar más. Esta celda carga pesos preentrenados, pero no los modifica ni entrena.

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_options = {'torch_dtype': torch.float16} if device.type == 'cuda' else {}

print('Dispositivo elegido:', device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, **model_options).to(device)
model.eval()
print('Modelo listo para generar.')

In [ ]:
for tone, prompt in prompts.items():
    raw_token_count = len(tokenizer(prompt, truncation=False)['input_ids'])
    used_token_count = min(raw_token_count, PHASE_CONFIG['max_input_tokens'])
    print(
        f'{tone}: {raw_token_count} tokens antes del límite; '
        f'hasta {used_token_count} se entregarán al modelo '
        f'(máximo {PHASE_CONFIG["max_input_tokens"]}).'
    )

## 5. Generar una salida por tono y reseña

Para cada reseña pedimos una respuesta con cada tono. `num_beams=4` y `do_sample=False` hacen que la generación sea determinista con estos parámetros: así intentamos que la diferencia principal entre resultados sea la instrucción de tono. Se generarán como máximo dos reseñas por tres tonos, es decir, seis resúmenes.

In [ ]:
generations = []
for review in reviews:
    for tone in PHASE_CONFIG['tone_labels']:
        prompt = build_prompt(review['review_text'], tone)
        summary = generate_summary(
            prompt=prompt,
            tokenizer=tokenizer,
            model=model,
            device=device,
            max_input_tokens=PHASE_CONFIG['max_input_tokens'],
            max_new_tokens=PHASE_CONFIG['max_new_tokens'],
        )
        generations.append(
            {
                'sample_index': review['sample_index'],
                'tone': tone,
                'rating': review.get('rating', ''),
                'reference_title': review.get('reference_title', ''),
                'review_text': review['review_text'],
                'generated_summary': summary,
            }
        )
        print(f"Reseña {review['sample_index']} | {tone}: {summary}\n")

print('Total de resúmenes generados:', len(generations))

## 6. Guardar y revisar resultados

Guardaremos la salida del notebook en `results/phase1_notebook_outputs.csv`, separada de la salida del script (`results/phase1_outputs.csv`). Así no se pisan entre sí. El archivo contiene la reseña y el resultado para poder juzgar fidelidad, claridad, concisión y tono. El título de la reseña es contexto, no una respuesta ideal.

In [ ]:
import csv
from html import escape
from IPython.display import HTML, display

output_path = PROJECT_ROOT / 'results' / 'phase1_notebook_outputs.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
fieldnames = list(generations[0].keys())
with output_path.open('w', encoding='utf-8', newline='') as output_file:
    writer = csv.DictWriter(output_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(generations)

table_rows = ''.join(
    '<tr>' + ''.join(f'<td>{escape(str(row[key]))}</td>' for key in ('tone', 'generated_summary')) + '</tr>'
    for row in generations
)
display(HTML('<table><thead><tr><th>Tono</th><th>Resumen generado</th></tr></thead><tbody>' + table_rows + '</tbody></table>'))
print('CSV guardado en:', output_path)

## 7. Evaluación cualitativa

Compara las tres variantes para una misma reseña y pregúntate:

- ¿Conservan los aspectos importantes de la experiencia?
- ¿Alguna inventa detalles o exagera la opinión?
- ¿Se distingue el tono pedido sin cambiar los hechos?
- ¿El resumen es breve y fácil de entender?

Todavía no calculamos ROUGE: el título de una reseña no es un resumen humano de referencia. Este experimento es una línea base por prompting, no fine-tuning ni RLHF. Para repetirlo con más ejemplos, cambia `N_SAMPLES` en la celda que carga las reseñas. Cada reseña genera una salida por tono.